# Sample downloads — inventory

Visual check of `data/samples/` from `manifest.csv`.

Re-download: `python dataloader/download_samples.py`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve() if Path.cwd().name == "dataloader" else Path(".").resolve()
OUT = ROOT / "data" / "samples"
MANIFEST = OUT / "manifest.csv"

assert MANIFEST.exists(), f"Missing {MANIFEST} — run: python dataloader/download_samples.py"
mf = pd.read_csv(MANIFEST)
mf["pdf_ok"] = mf["local_path"].notna() & mf["local_path"].map(lambda p: Path(p).exists() if pd.notna(p) else False)
mf["html_ok"] = mf["html_path"].notna() & mf["html_path"].map(lambda p: Path(p).exists() if pd.notna(p) else False)
mf["pdf_mb"] = mf["bytes"] / 1e6

print(f"rows={len(mf):,}  folders={mf['folder'].nunique()}  accessions={mf['accession'].nunique()}")
mf.head(3)

## Counts by folder / role / status

In [ ]:
summary = (
    mf.groupby(["folder", "doc_role"], dropna=False)
    .agg(
        n=("accession", "size"),
        ok=("status", lambda s: (s == "ok").sum()),
        pdf_ok=("pdf_ok", "sum"),
        html_ok=("html_ok", "sum"),
        pdf_mb=("pdf_mb", "sum"),
        entities=("entity_name", "nunique"),
    )
    .reset_index()
)
summary["pdf_mb"] = summary["pdf_mb"].round(1)
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
mf["folder"].value_counts().sort_values().plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_title("Files / folder")
axes[0].set_xlabel("n")

mf.groupby("year").size().plot.bar(ax=axes[1], color="#F58518", rot=0)
axes[1].set_title("By year")
axes[1].set_xlabel("")

mf["status"].value_counts().plot.bar(ax=axes[2], color="#54A24B", rot=0)
axes[2].set_title("Status")

fig.tight_layout()
plt.show()

## Forms & exhibit types

In [ ]:
print("form × folder")
display(pd.crosstab(mf["folder"], mf["form"]))

ex = mf[mf["folder"] == "8k_exhibits"]
if len(ex):
    print("\n8k_exhibits file_type")
    display(ex["file_type"].value_counts().rename_axis("file_type").reset_index(name="n"))

## Browse one folder

Change `FOLDER` and re-run.

In [ ]:
FOLDER = "8k"  # e.g. schedule_to, to_t, 14d9, 8k, 8k_exhibits, …

sub = mf[mf["folder"] == FOLDER].copy()
assert len(sub), f"no rows for folder={FOLDER!r}; choices={sorted(mf.folder.unique())}"

cols = [
    c
    for c in [
        "year", "form", "file_type", "entity_name", "accession", "filename",
        "pdf_mb", "pdf_ok", "html_ok", "status", "local_path",
    ]
    if c in sub.columns
]
display(sub[cols].sort_values(["year", "entity_name"]).reset_index(drop=True))

missing = sub[~sub["pdf_ok"] | ~sub["html_ok"]]
print(f"{FOLDER}: {len(sub)} rows, missing pdf/html={len(missing)}")
if len(missing):
    display(missing[["accession", "filename", "pdf_ok", "html_ok", "status", "error"]])

## Size distribution (PDF MB)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
order = mf.groupby("folder")["pdf_mb"].median().sort_values().index
data = [mf.loc[mf.folder == f, "pdf_mb"].dropna() for f in order]
ax.boxplot(data, tick_labels=list(order), vert=False)
ax.set_xlabel("PDF size (MB)")
ax.set_title("PDF size by folder")
fig.tight_layout()
plt.show()

print(mf.groupby("folder")["pdf_mb"].describe()[["count", "mean", "50%", "max"]].round(2).to_string())